# Quality - Validate river_discharge_daily

Valida integridad de `weather.silver.rating_curve_segments` / `river_discharge_daily` y mide
que tan bien reproduce la curva los caudales reales (plan Seccion 4.6):
1. claves unicas + tablas no vacias
2. control cruzado contra `Vazao_Adotada` (Bronze, gratuito, no depende de aforos descargados)
3. MAPE contra aforos reales, separando dentro de rango vs. extrapolados (mide el error de D3)
4. % de dias por estacion/anio segun `caudal_metodo`
5. distribucion de `distancia_fuera_rango_cm`
6. monotonicidad de la curva y continuidad entre segmentos consecutivos
7. saltos de caudal en los bordes de vigencia (mismo dia, curvas distintas)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SEGMENTS_TABLE = 'weather.silver.rating_curve_segments'
DISCHARGE_TABLE = 'weather.silver.river_discharge_daily'
BRONZE_LEVEL_TABLE = 'weather.bronze.ana_rio_uruguai'
BRONZE_DISCHARGE_TABLE = 'weather.bronze.ana_discharge_measurements'


def parse_decimal(column_name):
    return F.regexp_replace(F.trim(F.col(column_name).cast('string')), ',', '.').cast('double')


def assert_table_has_rows(table_name, filter_expr=None):
    df = spark.table(table_name)
    if filter_expr is not None:
        df = df.filter(filter_expr)
    row_count = df.count()
    print(f'{table_name}: {row_count} rows')
    if row_count == 0:
        raise ValueError(f'{table_name} has no rows')


def assert_unique(table_name, key_cols):
    df = spark.table(table_name)
    duplicates = df.groupBy(*key_cols).count().filter(F.col('count') > 1)
    duplicate_count = duplicates.count()
    print(f'{table_name} duplicate keys on {key_cols}: {duplicate_count}')
    if duplicate_count > 0:
        duplicates.show(20, truncate=False)
        raise ValueError(f'{table_name} has duplicate keys')


assert_table_has_rows(SEGMENTS_TABLE)
assert_table_has_rows(DISCHARGE_TABLE)
assert_unique(SEGMENTS_TABLE, ['codigoestacao', 'rating_curve_id', 'segment_number'])
assert_unique(DISCHARGE_TABLE, ['fecha', 'codigoestacao'])

## 1. Control cruzado contra `Vazao_Adotada` (Bronze)

La API de ANA ya trae un caudal adoptado para algunas estaciones/fechas -- sirve para
validar el calculo sin depender de los aforos descargados aparte (gratis, ya esta en Bronze).

In [ ]:
vazao_adotada = (
    spark.table(BRONZE_LEVEL_TABLE)
    .withColumn('fecha', F.to_date(F.to_timestamp('Data_Hora_Medicao')))
    .withColumn('vazao_adotada', parse_decimal('Vazao_Adotada'))
    .filter(F.col('vazao_adotada').isNotNull() & (F.col('vazao_adotada') > 0))
    .groupBy('fecha', 'codigoestacao')
    .agg(F.avg('vazao_adotada').alias('vazao_adotada_media'))
)

cross_check = (
    spark.table(DISCHARGE_TABLE)
    .filter(F.col('caudal_m3s').isNotNull())
    .join(vazao_adotada, ['fecha', 'codigoestacao'], 'inner')
    .withColumn('error_pct', F.abs(F.col('caudal_m3s') - F.col('vazao_adotada_media')) / F.col('vazao_adotada_media'))
)

n_cross = cross_check.count()
print(f'Vazao_Adotada disponible para cruzar: {n_cross} dias/estacion')
if n_cross > 0:
    cross_check.groupBy('caudal_metodo').agg(
        F.count('*').alias('n'), F.avg('error_pct').alias('mape'), F.expr('percentile(error_pct, 0.5)').alias('mediana'),
    ).orderBy(F.desc('n')).show(truncate=False)
else:
    print('No hay Vazao_Adotada en Bronze para el universo actual (esperado si las estaciones descargadas no la reportan).')

## 2. MAPE contra aforos reales, dentro de rango vs. extrapolados

Este es el numero clave de D3: cuanto se equivoca la extrapolacion de verdad, medido
contra mediciones de campo reales (no contra la propia curva).

In [ ]:
aforos = (
    spark.table(BRONZE_DISCHARGE_TABLE)
    .withColumn('fecha', F.to_date(F.to_timestamp('Data_Hora_Dado')))
    .withColumn('stage_cm', parse_decimal('Cota'))
    .withColumn('vazao_real', parse_decimal('Vazao'))
    .filter(F.col('stage_cm').isNotNull() & F.col('vazao_real').isNotNull() & (F.col('vazao_real') > 0) & F.col('fecha').isNotNull())
)

aforo_vs_calculado = (
    aforos.alias('a')
    .join(spark.table(DISCHARGE_TABLE).alias('d'), ['fecha', 'codigoestacao'], 'inner')
    .filter(F.col('d.caudal_m3s').isNotNull())
    .withColumn('error_pct', F.abs(F.col('d.caudal_m3s') - F.col('a.vazao_real')) / F.col('a.vazao_real'))
    .withColumn('bucket', F.when(F.col('d.caudal_metodo') == 'interpolado', F.lit('dentro_de_rango')).otherwise(F.lit('extrapolado')))
)

n_afo = aforo_vs_calculado.count()
print(f'Aforos comparables contra el caudal calculado: {n_afo}')
if n_afo > 0:
    aforo_vs_calculado.groupBy('bucket').agg(
        F.count('*').alias('n'), F.avg('error_pct').alias('mape'), F.expr('percentile(error_pct, 0.5)').alias('mediana'),
    ).show(truncate=False)
else:
    print('No hay aforos cruzables todavia (esperado hasta que el grupo A/B tenga aforos descargados).')

## 3. Cobertura por `caudal_metodo` (que porcion del dataset es interpolado vs. extrapolado)

In [ ]:
cobertura_metodo = (
    spark.table(DISCHARGE_TABLE)
    .withColumn('anio', F.year('fecha'))
    .groupBy('codigoestacao', 'anio')
    .pivot('caudal_metodo')
    .count()
    .fillna(0)
)
cobertura_metodo.orderBy('codigoestacao', 'anio').show(50, truncate=False)

print('Resumen global por caudal_metodo:')
total_rows = spark.table(DISCHARGE_TABLE).count()
(
    spark.table(DISCHARGE_TABLE)
    .groupBy('caudal_metodo')
    .count()
    .withColumn('pct', F.round(F.col('count') / F.lit(total_rows) * 100, 2))
    .orderBy(F.desc('count'))
    .show(truncate=False)
)

## 4. Distribucion de `distancia_fuera_rango_cm` (para calibrar un umbral de outlier)

In [ ]:
extrapolados = spark.table(DISCHARGE_TABLE).filter(F.col('caudal_extrapolado') == True)
n_extrap = extrapolados.count()
print(f'Filas extrapoladas: {n_extrap}')
if n_extrap > 0:
    extrapolados.select(
        F.expr('percentile(distancia_fuera_rango_cm, 0.5)').alias('p50'),
        F.expr('percentile(distancia_fuera_rango_cm, 0.9)').alias('p90'),
        F.expr('percentile(distancia_fuera_rango_cm, 0.99)').alias('p99'),
        F.max('distancia_fuera_rango_cm').alias('max'),
        F.sum(F.col('supera_aforo_maximo').cast('int')).alias('n_supera_aforo_maximo'),
    ).show(truncate=False)

## 5. Monotonicidad y continuidad de la curva por vigencia

In [ ]:
seg_window = Window.partitionBy('codigoestacao', 'rating_curve_id').orderBy('stage_min_cm')

continuidad = (
    spark.table(SEGMENTS_TABLE)
    .withColumn('next_stage_min_cm', F.lead('stage_min_cm').over(seg_window))
    .withColumn('gap_cm', F.col('next_stage_min_cm') - F.col('stage_max_cm'))
    .filter(F.col('next_stage_min_cm').isNotNull())
    .filter((F.col('gap_cm') > F.lit(1.0)) | (F.col('gap_cm') < F.lit(-1.0)))
)
n_gaps = continuidad.count()
print(f'Discontinuidades entre segmentos consecutivos de una misma vigencia (>1cm de hueco/solape): {n_gaps}')
if n_gaps > 0:
    continuidad.select('codigoestacao', 'rating_curve_id', 'stage_max_cm', 'next_stage_min_cm', 'gap_cm').show(30, truncate=False)

coef_window = Window.partitionBy('codigoestacao', 'rating_curve_id').orderBy('stage_min_cm')
monotonia = (
    spark.table(SEGMENTS_TABLE)
    .withColumn('q_en_stage_min', F.col('coefficient_a') * F.pow((F.col('stage_min_cm') / F.lit(100.0)) - F.col('coefficient_h0_m'), F.col('coefficient_n')))
    .withColumn('q_en_stage_max', F.col('coefficient_a') * F.pow((F.col('stage_max_cm') / F.lit(100.0)) - F.col('coefficient_h0_m'), F.col('coefficient_n')))
    .filter(F.col('q_en_stage_max') < F.col('q_en_stage_min'))
)
n_no_monotono = monotonia.count()
print(f'Segmentos donde Q no crece con la cota (posible error de coeficientes): {n_no_monotono}')
if n_no_monotono > 0:
    monotonia.select('codigoestacao', 'rating_curve_id', 'segment_number', 'q_en_stage_min', 'q_en_stage_max').show(30, truncate=False)

## 6. Saltos de caudal en los bordes de vigencia

Compara el caudal calculado el ultimo dia de una vigencia contra el primer dia de la
siguiente, con nivel similar -- un salto grande sugiere cambio de cero de escala.

In [ ]:
day_window = Window.partitionBy('codigoestacao').orderBy('fecha')
saltos = (
    spark.table(DISCHARGE_TABLE)
    .filter(F.col('caudal_m3s').isNotNull())
    .withColumn('prev_rating_curve_id', F.lag('rating_curve_id').over(day_window))
    .withColumn('prev_caudal', F.lag('caudal_m3s').over(day_window))
    .withColumn('prev_nivel', F.lag('nivel_media_cm').over(day_window))
    .filter(F.col('prev_rating_curve_id').isNotNull() & (F.col('rating_curve_id') != F.col('prev_rating_curve_id')))
    .withColumn('delta_nivel_cm', F.abs(F.col('nivel_media_cm') - F.col('prev_nivel')))
    .withColumn('delta_caudal_pct', F.abs(F.col('caudal_m3s') - F.col('prev_caudal')) / F.col('prev_caudal'))
    .filter((F.col('delta_nivel_cm') < F.lit(5.0)) & (F.col('delta_caudal_pct') > F.lit(0.15)))
)
n_saltos = saltos.count()
print(f'Saltos de vigencia con nivel casi igual (<5cm) pero caudal cambia >15%: {n_saltos}')
if n_saltos > 0:
    saltos.select('codigoestacao', 'fecha', 'prev_rating_curve_id', 'rating_curve_id', 'prev_nivel', 'nivel_media_cm', 'prev_caudal', 'caudal_m3s', 'delta_caudal_pct').show(30, truncate=False)